In [2]:
import sys
import os

from pathlib import Path

from dotenv import load_dotenv
from pyspark.sql import SparkSession
import pandas as pd
import numpy as np

load_dotenv()

sys.path.append("../../")
from src.lib.mlflow import MlflowHandler
from src.main import _ensure_java_home
mlflow_handler = MlflowHandler()

In [3]:
spark_app_name = os.getenv("SPARK_APP_NAME")
spark_master_url = os.getenv("SPARK_MASTER_URL")
postgres_url = os.getenv("POSTGRES_URL")
postgres_user = os.getenv("POSTGRES_USER")
postgres_password = os.getenv("POSTGRES_PASSWORD")
_ensure_java_home()

required = {
        "POSTGRES_URL": postgres_url,
        "POSTGRES_USER": postgres_user,
        "POSTGRES_PASSWORD": postgres_password,
    }

spark = (
        SparkSession.builder.appName(spark_app_name)
        .master(spark_master_url)
        .config("spark.jars.packages", "org.postgresql:postgresql:42.7.8")
        .config("spark.ui.showConsoleProgress", "false")
        .config("spark.sql.adaptive.enabled", "true")
        .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
        .getOrCreate()
)

25/11/03 19:01:32 WARN Utils: Your hostname, MacBook-Air-de-Yose.local resolves to a loopback address: 127.0.0.1; using 10.48.66.31 instead (on interface en0)
25/11/03 19:01:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/Users/yosesotomayor/Code/store/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/yosesotomayor/.ivy2/cache
The jars for the packages stored in: /Users/yosesotomayor/.ivy2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e9620764-77b1-4f91-a4d0-fffd6baa2797;1.0
	confs: [default]
	found org.postgresql#postgresql;42.7.8 in central
	found org.checkerframework#checker-qual;3.49.5 in central
:: resolution report :: resolve 234ms :: artifacts dl 13ms
	:: modules in use:
	org.checkerframework#checker-qual;3.49.5 from central in [default]
	org.postgresql#postgresql;42.7.8 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   2   |   0   |   0   |   0   ||   2   |   0   |
	--------------------------

In [4]:
missing = [k for k, v in required.items() if not v]
if missing:
        raise RuntimeError(
            f"Missing required environment variables for JDBC connection: {', '.join(missing)}"
        )

jdbc_options = {
        "url": str(postgres_url),
        "dbtable": "articles",
        "user": str(postgres_user),
        "password": str(postgres_password),
        "driver": "org.postgresql.Driver",
    }

reader = spark.read.format("jdbc")
for k, v in jdbc_options.items():
        reader = reader.option(k, v)

df = reader.load()

df.show()

+----------+------------+--------------------+---------------+-----------------+------------------+-----------------------+-------------------------+-----------------+-----------------+-------------------------+---------------------------+--------------------------+----------------------------+-------------+----------------+----------+--------------------+--------------+----------------+----------+--------------------+----------------+------------------+--------------------+
|article_id|product_code|           prod_name|product_type_no|product_type_name|product_group_name|graphical_appearance_no|graphical_appearance_name|colour_group_code|colour_group_name|perceived_colour_value_id|perceived_colour_value_name|perceived_colour_master_id|perceived_colour_master_name|department_no| department_name|index_code|          index_name|index_group_no|index_group_name|section_no|        section_name|garment_group_no|garment_group_name|         detail_desc|
+----------+------------+---------------

In [5]:
jdbc_options_2 = {
        "url": str(postgres_url),
        "dbtable": "customers",
        "user": str(postgres_user),
        "password": str(postgres_password),
        "driver": "org.postgresql.Driver",
    }

reader_2 = spark.read.format("jdbc")
for k, v in jdbc_options_2.items():
        reader_2 = reader_2.option(k, v)

df_customers = reader_2.load()

df_customers.show()

+--------------------+----+------+------------------+----------------------+---+--------------------+----+-----+-------------+----------------+----------+
|         customer_id|  fn|active|club_member_status|fashion_news_frequency|age|         postal_code|name|email|password_hash|is_authenticated|created_at|
+--------------------+----+------+------------------+----------------------+---+--------------------+----+-----+-------------+----------------+----------+
|f55a003032a8c8f81...|NULL|  NULL|        PRE-CREATE|                  NONE| 36|86ab807568b3dd0a9...|NULL| NULL|         NULL|           false|      NULL|
|f55a0c2cc5df55162...|NULL|  NULL|            ACTIVE|                  NONE| 20|f084a0d336e36861b...|NULL| NULL|         NULL|           false|      NULL|
|f55a13eeb44c04f22...|NULL|  NULL|            ACTIVE|                  NONE| 27|a29ed3dd3b856183d...|NULL| NULL|         NULL|           false|      NULL|
|f55a245f40d0b8690...|NULL|  NULL|            ACTIVE|                 

In [11]:
df_customers_pandas = df_customers.toPandas()

In [6]:
df_pandas = df.toPandas()

In [7]:
pd.set_option('display.max_columns', None)
df_pandas

,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,perceived_colour_value_id,perceived_colour_value_name,perceived_colour_master_id,perceived_colour_master_name,department_no,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,692844001,692844,TP Niclas low price jogger,272,Trousers,Garment Lower body,1010001,All over pattern,73,Dark Blue,4,Dark,2,Blue,7656,Kids Boy Trouser,H,Children Sizes 92-140,4,Baby/Children,46,Kids Boy,1009,Trousers,Pull-on trousers in a cotton weave with an ela...
1,692844002,692844,TP Niclas low price jogger,272,Trousers,Garment Lower body,1010016,Solid,8,Dark Grey,4,Dark,12,Grey,7656,Kids Boy Trouser,H,Children Sizes 92-140,4,Baby/Children,46,Kids Boy,1009,Trousers,Pull-on trousers in a cotton weave with an ela...
2,692844006,692844,TP Niclas low price jogger,272,Trousers,Garment Lower body,1010016,Solid,14,Dark Beige,2,Medium Dusty,11,Beige,7656,Kids Boy Trouser,H,Children Sizes 92-140,4,Baby/Children,46,Kids Boy,1009,Trousers,Pull-on trousers in a cotton weave with an ela...
3,692844008,692844,TP Niclas low price jogger,272,Trousers,Garment Lower body,1010016,Solid,73,Dark Blue,4,Dark,2,Blue,7656,Kids Boy Trouser,H,Children Sizes 92-140,4,Baby/Children,46,Kids Boy,1009,Trousers,Pull-on trousers in a cotton weave with an ela...
4,692844009,692844,TP Niclas low price jogger,272,Trousers,Garment Lower body,1010016,Solid,73,Dark Blue,2,Medium Dusty,2,Blue,7656,Kids Boy Trouser,H,Children Sizes 92-140,4,Baby/Children,46,Kids Boy,1009,Trousers,Pull-on trousers in a cotton weave with an ela...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105537,692842013,692842,TP Pirece chinos,272,Trousers,Garment Lower body,1010016,Solid,43,Dark Red,7,Medium,18,Red,7656,Kids Boy Trouser,H,Children Sizes 92-140,4,Baby/Children,46,Kids Boy,1009,Trousers,"Chinos in cotton twill with an adjustable, ela..."
105538,692843001,692843,Pierce 2 pack,272,Trousers,Garment Lower body,1010016,Solid,73,Dark Blue,4,Dark,2,Blue,7656,Kids Boy Trouser,H,Children Sizes 92-140,4,Baby/Children,46,Kids Boy,1009,Trousers,"Trousers in a cotton weave with an adjustable,..."
105539,692843002,692843,Pierce 2 pack,272,Trousers,Garment Lower body,1010016,Solid,42,Red,7,Medium,18,Red,7656,Kids Boy Trouser,H,Children Sizes 92-140,4,Baby/Children,46,Kids Boy,1009,Trousers,"Trousers in a cotton weave with an adjustable,..."
105540,692843004,692843,Pierce (TVP) 2-p,272,Trousers,Garment Lower body,1010016,Solid,73,Dark Blue,4,Dark,2,Blue,7656,Kids Boy Trouser,H,Children Sizes 92-140,4,Baby/Children,46,Kids Boy,1009,Trousers,"Trousers in a cotton weave with an adjustable,..."


In [12]:
df_customers_pandas

,customer_id,fn,active,club_member_status,fashion_news_frequency,age,postal_code,name,email,password_hash,is_authenticated,created_at
0,f55a003032a8c8f810a465f64bf9e36bafd896d29524bc...,NaN,NaN,PRE-CREATE,NONE,36.0,86ab807568b3dd0a9cea497ac473d7910a839b411e774d...,None,None,None,False,NaT
1,f55a0c2cc5df55162161cce2f134980b3d0bdf9af5982d...,NaN,NaN,ACTIVE,NONE,20.0,f084a0d336e36861b4b1b7a7d4ad39df697b79683c01f4...,None,None,None,False,NaT
2,f55a13eeb44c04f22783670cadab66c56d38750a4beffc...,NaN,NaN,ACTIVE,NONE,27.0,a29ed3dd3b856183db3a6dc8fc6a92a2bc1b1a627874d0...,None,None,None,False,NaT
3,f55a245f40d0b86905b4f615ca25e1c34f1bc5d9fabc39...,NaN,NaN,ACTIVE,NONE,53.0,5ba37477d8e5c3af86682c81f0018ac6bb85966a2fc757...,None,None,None,False,NaT
4,f55a2a5115aae273613a9a730ec26f141d79d9272409a5...,NaN,NaN,ACTIVE,NONE,25.0,08ee43710825ec81cc002e340689feb4f7ee320615a4b0...,None,None,None,False,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...
1371977,f559809eebef0671d5b5be737fabd134c913a29df98acb...,1.0,1.0,ACTIVE,Regularly,27.0,2c29ae653a9282cce4151bd87643c907644e09541abc28...,None,None,None,False,NaT
1371978,f5598e6e593f03740cbef42776cbf2dc20f379e8bebdb1...,1.0,1.0,ACTIVE,Regularly,24.0,2c29ae653a9282cce4151bd87643c907644e09541abc28...,None,None,None,False,NaT
1371979,f559ba6cd345bbe6f44547296bc03c906024441a6e5b33...,NaN,NaN,PRE-CREATE,NONE,NaN,502a058326ac4e090a3d3e1ffb53388f7c3713ad99bbe6...,None,None,None,False,NaT
1371980,f559beab8ebb0ec81ae812f6519cde4274250cf121f20e...,1.0,1.0,ACTIVE,Regularly,49.0,e0f349c59150ee8ab30f647c9b3ea38a1ef04f9494e25c...,None,None,None,False,NaT


In [14]:
df_transactions_pandas = pd.read_csv("/Users/yosesotomayor/Desktop/data_store/transactions_train.csv")

In [8]:
df_pandas['product_type_name'].nunique()

131

In [16]:
df_transactions_pandas = df_transactions_pandas[['customer_id', 'article_id']]

In [18]:
df_transactions_pandas.set_index(['customer_id'], inplace=True)

In [21]:
inter_ = df_transactions_pandas.index.intersection(df_customers_pandas['customer_id'])

In [25]:
df_customers_pandas

,customer_id,fn,active,club_member_status,fashion_news_frequency,age,postal_code,name,email,password_hash,is_authenticated,created_at
0,f55a003032a8c8f810a465f64bf9e36bafd896d29524bc...,NaN,NaN,PRE-CREATE,NONE,36.0,86ab807568b3dd0a9cea497ac473d7910a839b411e774d...,None,None,None,False,NaT
1,f55a0c2cc5df55162161cce2f134980b3d0bdf9af5982d...,NaN,NaN,ACTIVE,NONE,20.0,f084a0d336e36861b4b1b7a7d4ad39df697b79683c01f4...,None,None,None,False,NaT
2,f55a13eeb44c04f22783670cadab66c56d38750a4beffc...,NaN,NaN,ACTIVE,NONE,27.0,a29ed3dd3b856183db3a6dc8fc6a92a2bc1b1a627874d0...,None,None,None,False,NaT
3,f55a245f40d0b86905b4f615ca25e1c34f1bc5d9fabc39...,NaN,NaN,ACTIVE,NONE,53.0,5ba37477d8e5c3af86682c81f0018ac6bb85966a2fc757...,None,None,None,False,NaT
4,f55a2a5115aae273613a9a730ec26f141d79d9272409a5...,NaN,NaN,ACTIVE,NONE,25.0,08ee43710825ec81cc002e340689feb4f7ee320615a4b0...,None,None,None,False,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...
1371977,f559809eebef0671d5b5be737fabd134c913a29df98acb...,1.0,1.0,ACTIVE,Regularly,27.0,2c29ae653a9282cce4151bd87643c907644e09541abc28...,None,None,None,False,NaT
1371978,f5598e6e593f03740cbef42776cbf2dc20f379e8bebdb1...,1.0,1.0,ACTIVE,Regularly,24.0,2c29ae653a9282cce4151bd87643c907644e09541abc28...,None,None,None,False,NaT
1371979,f559ba6cd345bbe6f44547296bc03c906024441a6e5b33...,NaN,NaN,PRE-CREATE,NONE,NaN,502a058326ac4e090a3d3e1ffb53388f7c3713ad99bbe6...,None,None,None,False,NaT
1371980,f559beab8ebb0ec81ae812f6519cde4274250cf121f20e...,1.0,1.0,ACTIVE,Regularly,49.0,e0f349c59150ee8ab30f647c9b3ea38a1ef04f9494e25c...,None,None,None,False,NaT


In [27]:
len(inter_)

1362281

In [28]:
df_correcto = df_transactions_pandas.loc[inter_]


1362281

In [29]:
df_correcto.index.nunique()

1362281

In [31]:
df_correcto[~df_correcto.index.duplicated(keep='first')]

,article_id
customer_id,
000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318,663713001
00007d2de826758b65a93dd24ce629ed66842531df6699338c5570910a014cc2,505221004
00083cda041544b2fbb0e0d2905ad17da7cf1007526fb4c73235dccbbc132280,688873012
0008968c0d451dbc5a9968da03196fe20051965edde7413775c4eb3be9abe9c2,531310002
000aa7f0dc06cd7174389e76c9e132a67860c5f65f970699daccc14425ac31a8,501820043
...,...
fe99a0069d6b3c64c2707d0ce53b9311540917471d82df6e0db01dcc057bc90c,867969008
fecc5f77b5f7ee4570efde9ab05ec94d0de2bf80efb4f6fe55c8ea9c608b9b0c,915611003
fece2f68864c311a0b5208e2eb735b3dcde7e41461d327bdd033014a27168ce8,756322001


In [32]:
df_correcto

,article_id
customer_id,
000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318,663713001
000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318,541518023
000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318,663713001
000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318,578020002
000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318,723529001
...,...
fee56cc5315dafb35a4490ccc6f711092cae913550c83251955178bdd1bc6be3,903647001
fee56cc5315dafb35a4490ccc6f711092cae913550c83251955178bdd1bc6be3,903647001
ff5b8a8b26bf93a66290e9bd1b73393ac6a58968a7851941508cce49f5dfa469,913597001


In [ ]:
q